# GST Invoice Compliance Checker — OpenEnv Quickstart

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/VelrajMurugesan/OpenEnv-Hackathon/blob/master/notebooks/quickstart.ipynb)
[![HF Space](https://img.shields.io/badge/Live%20Space-VelrajMurugesan%2Fgst--invoice--compliance--checker-blue)](https://huggingface.co/spaces/VelrajMurugesan/gst-invoice-compliance-checker)

This notebook connects to the **live OpenEnv environment** running on Hugging Face Spaces and walks through the full agent loop end-to-end:

1. `reset()` — start a task and receive the invoices to audit
2. `step(flag_issue)` — flag compliance violations
3. `step(approve)` — mark clean invoices as compliant
4. `step(submit_report)` — finalize and receive the grader score

No GPU, no API key, no local install required. Just run the cells in order.

In [ ]:
!pip install -q httpx

In [ ]:
import json

import httpx

ENV_URL = "https://velrajmurugesan-gst-invoice-compliance-checker.hf.space"

client = httpx.Client(base_url=ENV_URL, timeout=60)

# Health check
print(client.get("/").json())

## 1. Inspect the available tasks

The environment ships with **10 tasks** across 3 difficulty tiers.

In [ ]:
tasks = client.get("/tasks").json()
for t in tasks:
    print(f"  [{t['difficulty']:6s}] {t['task_id']:9s} — {t['name']} ({t['num_invoices']} invoices)")

## 2. Reset to a task

We'll start with `easy_1` — a single B2B invoice with three missing mandatory fields.

In [ ]:
state = client.post("/reset", json={"task_id": "easy_1"}).json()
print(f"Task: {state['task_id']}")
print(f"Description: {state['task_description']}")
print(f"Difficulty: {state['difficulty']}")
print(f"Invoices: {len(state['invoices'])}")
print()
print("First invoice (truncated):")
inv = state["invoices"][0]
print(f"  invoice_id      = {inv['invoice_id']}")
print(f"  invoice_number  = {inv['invoice_number']!r}     <-- MISSING")
print(f"  supplier_gstin  = {inv['supplier_gstin']}")
print(f"  recipient_name  = {inv['recipient_name']!r}     <-- MISSING")
print(f"  recipient_gstin = {inv['recipient_gstin']!r}    <-- MISSING (B2B requires this)")

## 3. Flag the three missing-field violations

An agent would normally call an LLM here to discover the issues. For this notebook we hand-code the three findings the auditor should report.

In [ ]:
findings = [
    ("invoice_number",  "missing_field", "critical", "Invoice number is missing"),
    ("recipient_name",  "missing_field", "critical", "Recipient name is missing"),
    ("recipient_gstin", "missing_field", "critical", "B2B invoice is missing recipient GSTIN"),
]

for field, category, severity, description in findings:
    response = client.post("/step", json={
        "action": {
            "action": "flag_issue",
            "invoice_id": "INV-E1-001",
            "field": field,
            "category": category,
            "severity": severity,
            "description": description,
        }
    }).json()
    print(f"  flagged {field:18s} reward={response['reward']:.4f}")

## 4. Submit the final report

`submit_report` ends the session and returns the grader's F1 score.

In [ ]:
result = client.post("/step", json={"action": {"action": "submit_report"}}).json()

print(f"done:  {result['done']}")
print(f"score: {result['state']['score']}")
print()
details = result["info"]["grader_result"]["details"]
print("Grader breakdown:")
for key in ("precision", "recall", "true_positives", "false_positives", "missed_issues"):
    print(f"  {key:18s} = {details.get(key)}")

## 5. Try the adversarial task (`hard_4`)

`hard_4` is a precision test: 3 invoices that look suspicious but only **1** has a real GST violation. The other two are traps. A naive agent that flags everything will tank its precision score.

In [ ]:
state = client.post("/reset", json={"task_id": "hard_4"}).json()
print(f"Task: {state['task_id']} — {state['task_description']}")
print()
for inv in state["invoices"]:
    print(f"  {inv['invoice_id']}: {inv['supplier_name']} -> {inv['recipient_name']}")

In [ ]:
# Perfect agent: flag the real issue, approve the two traps, submit.
client.post("/step", json={
    "action": {
        "action": "flag_issue",
        "invoice_id": "INV-H4-003",
        "field": "recipient_state_code",
        "category": "inconsistency",
        "severity": "major",
        "description": "Recipient GSTIN prefix '07' (Delhi) does not match declared state '27' (Maharashtra)",
    }
})
client.post("/step", json={"action": {"action": "approve", "invoice_id": "INV-H4-001"}})
client.post("/step", json={"action": {"action": "approve", "invoice_id": "INV-H4-002"}})
result = client.post("/step", json={"action": {"action": "submit_report"}}).json()

print(f"hard_4 score: {result['state']['score']}")
print(f"precision:    {result['info']['grader_result']['details']['precision']}")
print(f"recall:       {result['info']['grader_result']['details']['recall']}")

## What's next

- Read [`README.md`](https://github.com/VelrajMurugesan/OpenEnv-Hackathon) for the full task taxonomy and grader design.
- Run the multi-model leaderboard locally with `uv run python benchmark.py`.
- Train your own agent against this environment using TRL's `GRPOTrainer` or any RL framework that supports OpenAI-compatible HTTP environments.